# BB84 vs CV-QKD: Headline Protocol Comparison

Generated from the public notebook builder for reproducible analysis.

Notebook 06 closes the project. No new physics modules are added; we combine everything from the earlier notebooks into a single headline figure that compares the BB84 and CV-QKD secure-key rates on the same axes under explicit, reproducible parameter sets.

## What this comparison can and cannot claim

* Both curves are *asymptotic* secure-key rates under stated parameter assumptions. **Finite-key effects are not included.**
* BB84 rates are reported **per emitted pulse**; CV-QKD rates are reported **per coherent-state symbol**. These are related channel-use normalisations &mdash; *not* a direct hardware throughput comparison.
* Cutoff distances and any "winner" language depend on the specific parameter set. Changing detector efficiency, excess noise, dark counts, or reconciliation efficiency changes the relative ordering.
* The defensible claim is that this repository implements and validates comparable DV/CV-QKD asymptotic models under stated assumptions; it does not claim that one protocol universally beats the other.

## Stated parameters

**BB84** (asymptotic simplified, idealised single-photon source):
* $\mu = 0.1$, $\eta_\mathrm{det} = 0.2$, $p_\mathrm{dark} = 10^{-6}$, $e_\mathrm{det} = 0$
* $f_\mathrm{ec} = 1.16$, $\alpha = 0.2$ dB/km
* rate normalisation: per emitted pulse

**CV-QKD GG02** (asymptotic collective Gaussian attack, untrusted-detector model):
* $V_A = 20$ shot-noise units, $\xi = 0.01$
* $\eta_\mathrm{det} = 0.6$, $\beta = 0.95$, $\alpha = 0.2$ dB/km
* rate normalisation: per coherent-state symbol

## 1. Bootstrap and imports

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise RuntimeError('Could not find project root')


PROJECT_ROOT = find_project_root()
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


Project root: C:\Users\COWLAR\projects\qkd-protocol-simulator


In [2]:
import os

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from src.channel import (
    bb84_key_rate, fiber_transmittance, qber_channel,
)
from src.cvqkd import (
    cvqkd_holevo_bound_homodyne,
    cvqkd_key_rate,
    cvqkd_mutual_info_homodyne,
)
from src.info_theory import binary_entropy
from src.plotting import semilogy_positive

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

L = np.linspace(0.0, 300.0, 1001)


## 2. Compute both key-rate curves

These call the existing Notebook 03 / Notebook 04/05 implementations verbatim &mdash; the headline figure is just a combined result.

In [3]:
rate_bb84 = bb84_key_rate(L)
rate_cvqkd = cvqkd_key_rate(L)

print(f'BB84   K(0)   = {float(bb84_key_rate(0)):.6e} bits / pulse')
print(f'CV-QKD K(0)   = {float(cvqkd_key_rate(0)):.6e} bits / symbol')
print(f'BB84   K(50)  = {float(bb84_key_rate(50)):.6e} bits / pulse')
print(f'CV-QKD K(50)  = {float(cvqkd_key_rate(50)):.6e} bits / symbol')
print(f'BB84   K(150) = {float(bb84_key_rate(150)):.6e} bits / pulse')
print(f'CV-QKD K(150) = {float(cvqkd_key_rate(150)):.6e} bits / symbol')


BB84   K(0)   = 9.984011e-03 bits / pulse
CV-QKD K(0)   = 4.524346e-01 bits / symbol
BB84   K(50)  = 9.875977e-04 bits / pulse
CV-QKD K(50)  = 9.236829e-03 bits / symbol
BB84   K(150) = 4.661664e-06 bits / pulse
CV-QKD K(150) = 0.000000e+00 bits / symbol


## 3. Cutoff distances via Brent root-finding 

We find the cutoff by running `scipy.optimize.brentq` on the **unclamped** security expressions &mdash; $1 - h(E) - f_{ec}h(E)$ for BB84 and $\beta\,I(A:B) - \chi(B:E)$ for CV-QKD &mdash; **not** on the clamped key-rate curves, which lack a sign change.

Nothing in the table below is hard-coded.

In [4]:
def bb84_unclamped_privacy_bracket(
    L_km, mu=0.1, eta_det=0.2, p_dark=1e-6, e_det=0.0,
    f_ec=1.16, alpha_db_per_km=0.2,
):
    qber = qber_channel(
        L_km, mu=mu, eta_det=eta_det, p_dark=p_dark,
        alpha_dB=alpha_db_per_km, e_det=e_det,
    )
    return float(1.0 - binary_entropy(qber) - f_ec * binary_entropy(qber))


def cvqkd_unclamped_key_expression(
    L_km, V_A=20.0, xi=0.01, beta=0.95, eta_det=0.6,
    alpha_db_per_km=0.2,
):
    eta_ch = float(fiber_transmittance(L_km, alpha_dB=alpha_db_per_km))
    eta = eta_ch * eta_det
    if eta <= 0.0:
        return -np.inf
    I_AB = float(cvqkd_mutual_info_homodyne(V_A=V_A, eta=eta, xi=xi))
    chi_BE = float(cvqkd_holevo_bound_homodyne(V_A=V_A, eta=eta, xi=xi))
    return beta * I_AB - chi_BE


def cutoff_from_unclamped(expr, L_grid):
    vals = np.array([expr(float(l)) for l in L_grid])
    finite = np.isfinite(vals)
    if not np.any(finite):
        raise ValueError('No finite values in cutoff search')
    if np.all(vals[finite] > 0):
        return float(L_grid[-1])
    crossing = np.where(finite & (vals <= 0))[0]
    if len(crossing) == 0:
        raise ValueError('No finite zero crossing found')
    idx = int(crossing[0])
    if idx == 0:
        return float(L_grid[0])
    return brentq(expr, float(L_grid[idx - 1]), float(L_grid[idx]))


L_max_bb84 = cutoff_from_unclamped(bb84_unclamped_privacy_bracket, L)
L_max_cvqkd = cutoff_from_unclamped(cvqkd_unclamped_key_expression, L)

print(f'BB84   cutoff: {L_max_bb84:.2f} km')
print(f'CV-QKD cutoff: {L_max_cvqkd:.2f} km')


BB84   cutoff: 169.38 km
CV-QKD cutoff: 74.19 km
